In [0]:
orders_full = spark.table("olist_project.bronze.orders")

# Split into 3 batches by purchase date (adjust dates to your actual data range)
batch1 = orders_full.filter("order_purchase_timestamp < '2017-01-01'")
batch2 = orders_full.filter("order_purchase_timestamp >= '2017-01-01' AND order_purchase_timestamp < '2017-07-01'")
batch3 = orders_full.filter("order_purchase_timestamp >= '2017-07-01'")

print(f"Batch1: {batch1.count()}, Batch2: {batch2.count()}, Batch3: {batch3.count()}")

# Write each batch as a separate CSV file into a staging folder in the volume
staging_path = "/Volumes/olist_project/default/raw_data/order_batches"
dbutils.fs.mkdirs(staging_path)
batch1.toPandas().to_csv(f"{staging_path}/orders_batch1.csv", index=False)
batch2.toPandas().to_csv(f"{staging_path}/orders_batch2.csv", index=False)
batch3.toPandas().to_csv(f"{staging_path}/orders_batch3.csv", index=False)

# Move batch1 only into the volume for now (simulating "day 1" data)
dbutils.fs.cp(f"{staging_path}/orders_batch1.csv",
              "/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch1.csv")

Batch1: 329, Batch2: 14611, Batch3: 84501


True

In [0]:
display(dbutils.fs.ls("/Volumes/olist_project/default/raw_data/incoming_orders/"))

path,name,size,modificationTime
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch1.csv,orders_batch1.csv,61850,1790676267000


In [0]:
display(dbutils.fs.ls("/Volumes/olist_project/default/raw_data/order_batches/"))

path,name,size,modificationTime
dbfs:/Volumes/olist_project/default/raw_data/order_batches/orders_batch1.csv,orders_batch1.csv,61850,1790676262000
dbfs:/Volumes/olist_project/default/raw_data/order_batches/orders_batch2.csv,orders_batch2.csv,2812414,1790676263000
dbfs:/Volumes/olist_project/default/raw_data/order_batches/orders_batch3.csv,orders_batch3.csv,16322929,1790676266000


In [0]:
checkpoint_path = "/Volumes/olist_project/default/raw_data/_checkpoints/orders_incremental"
bronze_incremental_path = "olist_project.bronze.orders_incremental"

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .load("/Volumes/olist_project/default/raw_data/incoming_orders/"))

(df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_incremental_path))

In [0]:
display(spark.table("olist_project.bronze.orders_incremental"))
print(spark.table("olist_project.bronze.orders_incremental").count())

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_ingested_at,_rescued_data
d3c8851a6651eeff2f73b0e011ac45d0,957f8e082185574de25992dc659ebbc0,processing,2016-10-05 22:44:13,2016-10-06 15:51:05,null,null,2016-12-09,2026-09-29 05:43:32.008688,null
cbbb524a0e1646aa6cf7a3c0bbe517ad,dacb079d55ffb1d3955c5d923df3ebb7,delivered,2016-10-05 07:31:49,2016-10-06 02:46:24,2016-10-10 02:46:24,2016-10-16 14:36:00,2016-12-07,2026-09-29 05:43:32.008688,null
ac2b7c522d811acba0aa270ed3e112e4,ef21aebbb093a6db29ccc6aa0b89c347,delivered,2016-10-05 15:08:00,2016-10-06 15:44:26,2016-10-10 15:44:27,2016-10-13 15:44:27,2016-11-29,2026-09-29 05:43:32.008688,null
7033745709b7cf1bac7d2533663592de,7f0ca17bb33b230b47459437cf0682c7,delivered,2016-10-04 14:13:22,2016-10-04 14:46:48,2016-10-08 14:46:49,2016-10-11 14:46:49,2016-11-30,2026-09-29 05:43:32.008688,null
5cd498954e2b37d71b315166809b4bd7,ff1a56726b7ea149c7423865609cc0c8,delivered,2016-10-07 00:54:40,2016-10-08 03:56:15,2016-10-25 11:35:52,2016-10-27 17:32:07,2016-12-01,2026-09-29 05:43:32.008688,null
dc5f6cd4492bbffe5bcda9b87856c9a5,2ebf822bbf33241c7bffb42235519f92,delivered,2016-10-05 13:12:43,2016-10-06 15:47:29,2016-10-21 16:34:58,2016-10-27 12:33:29,2016-11-29,2026-09-29 05:43:32.008688,null
ddaec6fff982b13e7e048b627a11d6da,68f4ad79cc0c2ad06e19088f5c00e9fa,canceled,2016-10-04 19:41:32,null,null,null,2016-11-16,2026-09-29 05:43:32.008688,null
cb2b577a4ebd84bf84ff3990084d490f,ebd67b458c0b8b23fa5d869779adc761,delivered,2016-10-05 17:11:49,2016-10-06 15:54:57,2016-10-30 15:14:54,2016-10-31 21:41:43,2016-11-29,2026-09-29 05:43:32.008688,null
3b2ca3293a7ce539ea2379d704fa37ce,06a70917afd2dcd59396e1eac836c646,delivered,2016-10-05 12:44:09,2016-10-06 15:47:17,2016-11-17 11:54:02,2016-11-17 19:53:49,2016-11-25,2026-09-29 05:43:32.008688,null
e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,null,null,2016-10-28,2026-09-29 05:43:32.008688,null


329


In [0]:
staging_path = "/Volumes/olist_project/default/raw_data/order_batches"
incoming_path = "/Volumes/olist_project/default/raw_data/incoming_orders"

dbutils.fs.cp(f"{staging_path}/orders_batch2.csv",
              f"{incoming_path}/orders_batch2.csv")

display(dbutils.fs.ls(incoming_path))

path,name,size,modificationTime
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch1.csv,orders_batch1.csv,61850,1790676267000
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch2.csv,orders_batch2.csv,2812414,1790677795000


In [0]:
total = spark.table("olist_project.bronze.orders_incremental").count()
print(f"Total rows after 2nd run: {total}")

Total rows after 2nd run: 14940


In [0]:
display(spark.sql("DESCRIBE HISTORY olist_project.bronze.orders_incremental"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-29T10:34:04.000Z,70688357809981,karuonline247@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> e8816503-2c1e-408d-94ed-795018e77fd7, epochId -> 1, statsOnLoad -> true)",null,List(4283308947654282),31633d68-abab-460e-be47-5bfb897cc903,0929-100145-13btzfgy-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 14611, numOutputBytes -> 822700, numAddedFiles -> 1)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-29T10:11:39.000Z,70688357809981,karuonline247@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> e8816503-2c1e-408d-94ed-795018e77fd7, epochId -> 0, statsOnLoad -> true)",null,List(4283308947654282),14bb539f-d8cd-4e9c-96f1-88810f6bfa78,0929-100145-13btzfgy-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 329, numOutputBytes -> 22359, numAddedFiles -> 1)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-29T10:11:29.000Z,70688357809981,karuonline247@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""13e29e3f-7c55-4a5c-92b9-284421dcabc5"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-40ce2fba-172a-4825-8e3f-53481eb1e208"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-b0d4cd6e-e428-4f2f-845b-866f775696ba""}, statsOnLoad -> false)",null,List(4283308947654282),14bb539f-d8cd-4e9c-96f1-88810f6bfa78,0929-100145-13btzfgy-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
staging_path = "/Volumes/olist_project/default/raw_data/order_batches"
incoming_path = "/Volumes/olist_project/default/raw_data/incoming_orders"

dbutils.fs.cp(f"{staging_path}/orders_batch3.csv",
              f"{incoming_path}/orders_batch3.csv")

display(dbutils.fs.ls(incoming_path))

path,name,size,modificationTime
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch1.csv,orders_batch1.csv,61850,1790676267000
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch2.csv,orders_batch2.csv,2812414,1790677795000
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch3.csv,orders_batch3.csv,16322929,1790678458000


In [0]:
total = spark.table("olist_project.bronze.orders_incremental").count()
print(f"Total rows after 3rd run: {total}")
# Expect 329 + 14611 + 84501 = 99441 (should now match the full orders table)

Total rows after 3rd run: 99441


In [0]:
display(spark.sql("DESCRIBE HISTORY olist_project.bronze.orders_incremental").select("version", "operation", "operationMetrics"))

version,operation,operationMetrics
3,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 84501, numOutputBytes -> 4901776, numAddedFiles -> 1)"
2,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 14611, numOutputBytes -> 822700, numAddedFiles -> 1)"
1,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 329, numOutputBytes -> 22359, numAddedFiles -> 1)"
0,CREATE TABLE,Map()


In [0]:
staging_path = "/Volumes/olist_project/default/raw_data/order_batches"
incoming_path = "/Volumes/olist_project/default/raw_data/incoming_orders"

dbutils.fs.cp(f"{staging_path}/orders_batch3.csv",
              f"{incoming_path}/orders_batch3.csv")

display(dbutils.fs.ls(incoming_path))

path,name,size,modificationTime
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch1.csv,orders_batch1.csv,61850,1790676267000
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch2.csv,orders_batch2.csv,2812414,1790677795000
dbfs:/Volumes/olist_project/default/raw_data/incoming_orders/orders_batch3.csv,orders_batch3.csv,16322929,1790746529000


In [0]:
total = spark.table("olist_project.bronze.orders_incremental").count()
print(f"Total rows after 3rd run: {total}")
# Expect 329 + 14611 + 84501 = 99441

Total rows after 3rd run: 99441


In [0]:
display(spark.sql("DESCRIBE HISTORY olist_project.bronze.orders_incremental").select("version", "operation", "operationMetrics"))

version,operation,operationMetrics
3,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 84501, numOutputBytes -> 4901776, numAddedFiles -> 1)"
2,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 14611, numOutputBytes -> 822700, numAddedFiles -> 1)"
1,STREAMING UPDATE,"Map(numRemovedFiles -> 0, numOutputRows -> 329, numOutputBytes -> 22359, numAddedFiles -> 1)"
0,CREATE TABLE,Map()


Simulated 3 daily batches of the orders table; Auto Loader with checkpointing processed each batch exactly once, reconstructing the full 99,441-row dataset across 3 incremental runs with zero duplicate processing, verified via Delta Lake's transaction history.

In [0]:
display(spark.sql("SELECT SUM(total_paid) as total_revenue FROM olist_project.gold.fact_orders"))

total_revenue
1.6008872119999027E7
